In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retail")

bronze = spark.table("workspace.retail.bronze_online_retail")
print("Total rows:", bronze.count())

In [0]:
display(bronze.groupBy("source_file").count())

In [0]:
display(
    bronze.withColumn("is_cancellation", F.col("invoice").startswith("C")
                      )
    .groupBy("is_cancellation")
    .agg(
        F.count("*").alias("rows"),
        F.min("quantity").alias("min_qty"),
        F.max("quantity").alias("max_qty"),
    )
)


In [0]:
total = bronze.count()
missing = bronze.filter(F.col("customer_id").isNull()).count()
print(f"Rows with no customer id : {missing} ({missing / total:.1%})")

In [0]:
odd_codes = (
    bronze
    .filter(~F.col("stock_code").rlike("^[0-9]{5}[A-Za-z]*$"))
    .groupBy("stock_code")
    .agg(
        F.count("*").alias("rows"),
        F.first("description").alias("sample_description"),
    )
    .orderBy(F.desc("rows"))
)
display(odd_codes.limit(40))

In [0]:
display(
    bronze.select(
        F.sum((F.col("price") < 0).cast("int")).alias("negative_price"),
        F.sum((F.col("price") == 0).cast("int")).alias("zero_price"),
        F.sum((F.col("quantity") == 0).cast("int")).alias("zero_quantity"),
        F.sum(
            ((F.col("quantity") < 0) & ~F.col("invoice").startswith("C")).cast("int")
        ).alias("negative_qty_not_cancellation"),
    )
)

In [0]:
business_cols = ["invoice", "stock_code", "description", "quantity",
                 "invoice_date", "price", "customer_id", "country"]

dupes = (
    bronze.groupBy(business_cols)
    .count()
    .filter(F.col("count") > 1)
)

print("Distinct duplicated row patterns:", dupes.count())
print("Extra rows caused by duplicates:",
      dupes.agg(F.sum(F.col("count") - 1)).first()[0])

In [0]:
display(
    bronze.groupBy("source_file")
    .agg(
        F.min("invoice_date").alias("first_date"),
        F.max("invoice_date").alias("last_date"),
    )
)

inv_1 = bronze.filter(F.col("source_file").contains("2009_2010")).select("invoice").distinct()
inv_2 = bronze.filter(F.col("source_file").contains("2010_2011")).select("invoice").distinct()

print("Invoices present in BOTH files:", inv_1.intersect(inv_2).count())

In [0]:
display(bronze.select(
    F.sum(F.col("description").isNull().cast("int")).alias("null_description"),
    F.sum((F.trim(F.col("description")) != F.col("description")).cast("int")).alias("untrimmed_description"),
))

display(bronze.groupBy("country").count().orderBy(F.desc("count")).limit(50))

In [0]:
business_cols = ["invoice", "stock_code", "description", "quantity",
                 "invoice_date", "price", "customer_id", "country"]

f1 = bronze.filter(F.col("source_file").contains("2009_2010"))
f2 = bronze.filter(F.col("source_file").contains("2010_2011"))

In [0]:
inv_both = f1.select("invoice").distinct().intersect(f2.select("invoice").distinct()).count()
rows_both = f1.select(business_cols).distinct().intersect(f2.select(business_cols).distinct()).count()

print("Invoices in BOTH files:", inv_both)
print("Identical rows in BOTH files:", rows_both)

win_start, win_end = "2010-12-01 00:00:00", "2010-12-09 23:59:59"

f1_win = f1.filter(F.col("invoice_date").between(win_start, win_end)).select(business_cols)
f2_win = f2.filter(F.col("invoice_date").between(win_start, win_end)).select(business_cols)

print("Rows in file 1 window:", f1_win.count())
print("Rows in file 2 window:", f2_win.count())
print("In file 1 but not in file 2:", f1_win.exceptAll(f2_win).count())
print("In file 2 but not in file 1:", f2_win.exceptAll(f1_win).count())

In [0]:
display(
    bronze.filter(F.col("invoice_date").between("2010-12-01 00:00:00", "2010-12-09 23:59:59"))
    .groupBy("source_file").count()
)

In [0]:
within_file = (
    bronze.groupBy("source_file", *business_cols)
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate patterns inside one file:", within_file.count())
print("Extra rows from within-file duplicates:",
      within_file.agg(F.sum(F.col("count") - 1)).first()[0])

In [0]:
display(bronze.filter(F.col("invoice").startswith("C") & (F.col("quantity") > 0)))

In [0]:
display(bronze.filter(F.col("price") < 0))

display(
    bronze.filter(F.col("price") == 0)
    .agg(
        F.count("*").alias("zero_price_rows"),
        F.sum(F.col("customer_id").isNull().cast("int")).alias("no_customer"),
        F.sum(F.col("description").isNull().cast("int")).alias("no_description"),
    )
)

display(
    bronze.filter(F.col("price") == 0)
    .select("invoice", "stock_code", "description", "quantity", "customer_id")
    .limit(15)
)

In [0]:
display(
    bronze.filter((F.col("quantity") < 0) & ~F.col("invoice").startswith("C"))
    .select("invoice", "stock_code", "description", "quantity", "price", "customer_id")
    .limit(15)
)

In [0]:
display(odd_codes.limit(100))